# 使用 RAGAS 評估 RAG 系統

RAGAS（Retrieval-Augmented Generation Assessment）是一套評估 LLM 應用程式的工具。對 RAG 而言，只確認「程式能回答」並不夠，我們還要分別檢查：

- Retriever 有沒有找到正確且足夠的內容？
- Generator 的回答是否忠於檢索內容？
- 回答是否符合人工準備的參考答案？

本練習使用 RAGAS 0.4 collections API，最後說明如何把結果接到 MLflow 做實驗追蹤。

## 評估流程

```text
測試問題
   ↓
Retriever → retrieved_contexts
   ↓
Generator → response
   ↓
RAGAS Judge ← reference
   ↓
逐題分數、理由、平均分數
   ↓
可選：記錄到 MLflow，比較不同模型與檢索設定
```

## 1. Import libraries

這份 notebook 以目前環境的 RAGAS 0.4 API 為準。舊教材中的 `ragas.metrics` legacy API 仍可能可用，但會逐步淘汰。

In [1]:
import os
from statistics import mean

import chromadb
import pandas as pd
import ragas
from IPython.display import display
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    ContextPrecision,
    ContextRecall,
    FactualCorrectness,
    Faithfulness,
)

print("RAGAS version:", ragas.__version__)

RAGAS version: 0.4.3


## 2. 建立 RAG Model、Embedding Model 與 Evaluator Model

RAGAS 的 evaluator 是 LLM-as-a-Judge。它和負責回答問題的模型可以不同：

- `OPENAI_MODEL`：RAG 產生答案的模型。
- `OPENAI_EMBEDDING_MODEL`：建立與查詢向量的模型。
- `RAGAS_EVALUATOR_MODEL`：負責評分的模型。

請先設定系統環境變數 `OPENAI_API_KEY`。自訂 API endpoint 可使用 `OPENAI_BASE_URL`。

In [2]:
api_key = os.environ["OPENAI_API_KEY"]
base_url = os.getenv("OPENAI_BASE_URL")
rag_model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
embedding_model_name = os.getenv(
    "OPENAI_EMBEDDING_MODEL",
    "text-embedding-3-small",
)
evaluator_model_name = os.getenv(
    "RAGAS_EVALUATOR_MODEL",
    "gpt-4o-mini",
)

chat_options = {
    "api_key": api_key,
    "model": rag_model_name,
    "temperature": 0.0,
}
embedding_options = {
    "api_key": api_key,
    "model": embedding_model_name,
}
evaluator_client_options = {"api_key": api_key}

if base_url:
    chat_options["base_url"] = base_url
    embedding_options["base_url"] = base_url
    evaluator_client_options["base_url"] = base_url

rag_llm = ChatOpenAI(**chat_options)
embeddings = OpenAIEmbeddings(**embedding_options)
evaluator_client = AsyncOpenAI(**evaluator_client_options)
evaluator_llm = llm_factory(
    evaluator_model_name,
    client=evaluator_client,
)

print("RAG model:      ", rag_model_name)
print("Embedding model:", embedding_model_name)
print("Evaluator model:", evaluator_model_name)

RAG model:       gpt-4o-mini
Embedding model: text-embedding-3-small
Evaluator model: gpt-4o-mini


## 3. 建立小型知識庫

真實專案通常會讀取 PDF、網頁或內部文件並切成 chunks。為了把重點放在評估，本例直接準備幾個短文件。

In [3]:
documents = [
    Document(
        page_content=(
            "LangGraph State 是節點之間共享的資料結構。"
            "每個 Node 可以讀取 State，並回傳要更新的欄位。"
        ),
        metadata={"source": "lesson-state"},
    ),
    Document(
        page_content=(
            "MemorySaver 是記憶體內的 checkpointer，會依 thread_id 保存 checkpoints。"
            "Python 程序結束後，MemorySaver 中的資料也會消失。"
        ),
        metadata={"source": "lesson-memory"},
    ),
    Document(
        page_content=(
            "Human-in-the-Loop 可用 interrupt_before 在敏感節點執行前暫停。"
            "人工同意後，以相同 thread_id 和 input=None 恢復流程。"
        ),
        metadata={"source": "lesson-hitl"},
    ),
    Document(
        page_content=(
            "ReAct Agent 會讓模型判斷是否使用工具。"
            "工具結果以 ToolMessage 回到模型，模型再決定下一步或產生答案。"
        ),
        metadata={"source": "lesson-react"},
    ),
    Document(
        page_content=(
            "MLflow Tracing 可以記錄一次 Agent 請求中的 LLM、Tool 與 Node spans。"
            "RAGAS 評估品質，MLflow 則保存實驗參數、分數、artifacts 與 traces。"
        ),
        metadata={"source": "lesson-observability"},
    ),
]

document_ids = [f"lesson-{index}" for index in range(1, len(documents) + 1)]

## 4. 建立 ChromaDB Retriever

本例使用記憶體內的 ChromaDB。`TOP_K` 是重要實驗參數：太小可能漏資料，太大則可能加入雜訊。

In [4]:
TOP_K = 2
chroma_client = chromadb.EphemeralClient()

vector_store = Chroma(
    client=chroma_client,
    collection_name="ragas-course",
    embedding_function=embeddings,
)
vector_store.add_documents(documents=documents, ids=document_ids)
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

print("Documents in vector store:", vector_store._collection.count())

Documents in vector store: 5


## 5. 建立基本 RAG Chain

In [5]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 LangGraph 課程助教。只能根據 Context 回答，"
            "不要加入 Context 沒提到的資訊。如果資料不足，請說不知道。"
            "回答請使用繁體中文，限制三句。",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)

rag_chain = rag_prompt | rag_llm | StrOutputParser()


def run_rag(question: str) -> dict:
    retrieved_documents = retriever.invoke(question)
    retrieved_contexts = [
        document.page_content for document in retrieved_documents
    ]
    context = "\n\n".join(retrieved_contexts)
    response = rag_chain.invoke(
        {"question": question, "context": context}
    )

    return {
        "user_input": question,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_sources": [
            document.metadata["source"] for document in retrieved_documents
        ],
        "response": response,
    }

## 6. 準備 Evaluation Dataset

高品質測試集應涵蓋真實問題、邊界案例與可能失敗的問題。`reference` 是人工確認過的理想答案，不是模型自己產生後直接當標準答案。

In [6]:
test_cases = [
    {
        "user_input": "MemorySaver 的資料會永久保存嗎？",
        "reference": (
            "不會。MemorySaver 將 checkpoints 保存在目前 Python 程序的記憶體中，"
            "程序結束後資料會消失。"
        ),
    },
    {
        "user_input": "如何讓 LangGraph 在發布前等待人工同意？",
        "reference": (
            "可用 interrupt_before 在發布節點前暫停，"
            "人工同意後以相同 thread_id 和 input=None 恢復。"
        ),
    },
    {
        "user_input": "RAGAS 和 MLflow 在評估流程中分別做什麼？",
        "reference": (
            "RAGAS 負責計算 RAG 品質指標；MLflow 保存實驗參數、"
            "分數、artifacts 與 traces，供比較和除錯。"
        ),
    },
]

In [7]:
evaluation_rows = []

for test_case in test_cases:
    row = run_rag(test_case["user_input"])
    row["reference"] = test_case["reference"]
    evaluation_rows.append(row)

generation_df = pd.DataFrame(evaluation_rows)
display(
    generation_df[
        ["user_input", "retrieved_sources", "response", "reference"]
    ]
)

,user_input,retrieved_sources,response,reference
0,MemorySaver 的資料會永久保存嗎？,"[lesson-memory, lesson-state]",MemorySaver 的資料在 Python 程序結束後會消失，因此不會永久保存。它僅在記...,不會。MemorySaver 將 checkpoints 保存在目前 Python 程序的記...
1,如何讓 LangGraph 在發布前等待人工同意？,"[lesson-state, lesson-hitl]",要讓 LangGraph 在發布前等待人工同意，可以在敏感節點使用 `interrupt_b...,可用 interrupt_before 在發布節點前暫停，人工同意後以相同 thread_i...
2,RAGAS 和 MLflow 在評估流程中分別做什麼？,"[lesson-observability, lesson-react]",RAGAS 主要負責評估品質，而 MLflow 則用來保存實驗參數、分數、artifacts...,RAGAS 負責計算 RAG 品質指標；MLflow 保存實驗參數、分數、artifacts...


## 7. 選擇 RAGAS Metrics

| Metric | 評估對象 | 需要的欄位 | 問的問題 |
|---|---|---|---|
| Faithfulness | Generator | response、retrieved_contexts | 回答中的主張能否由 context 支持？ |
| Context Precision | Retriever | user_input、reference、retrieved_contexts | 排名前面的 context 是否真的有用？ |
| Context Recall | Retriever | user_input、reference、retrieved_contexts | reference 所需資訊是否都有被找回？ |
| Factual Correctness | Generator | response、reference | 回答和參考答案的事實是否一致？ |

分數通常介於 0～1，越高越好，但不是絕對真理。LLM judge 仍可能有偏差，重要案例應人工抽查。

In [8]:
faithfulness_metric = Faithfulness(llm=evaluator_llm)
context_precision_metric = ContextPrecision(llm=evaluator_llm)
context_recall_metric = ContextRecall(llm=evaluator_llm)
factual_correctness_metric = FactualCorrectness(llm=evaluator_llm)

## 8. 先評估單一案例

RAGAS 0.4 的 `ascore()` 回傳 `MetricResult`。`value` 是分數；`reason` 雖然是可選欄位，但目前這四個 collections metrics 只回傳 `value`，因此直接讀取 `.reason` 會得到 `None`。一次 metric 可能在內部呼叫 evaluator LLM 多次，因此評估也會產生 token 成本。

In [9]:
sample = evaluation_rows[0]

faithfulness_result = await faithfulness_metric.ascore(
    user_input=sample["user_input"],
    response=sample["response"],
    retrieved_contexts=sample["retrieved_contexts"],
)

print("Score:       ", faithfulness_result.value)
print("RAGAS reason:", faithfulness_result.reason)  # 此版本預期為 None

Score:        1.0
RAGAS reason: None


## 9. 評估完整資料集

以下逐題執行四個 metrics，並補上容易閱讀的分數解讀。這些 `interpretation` 是依分數區間產生的教學提示，不是 RAGAS evaluator LLM 回傳的原生理由。資料量大時應考慮 rate limit、並行數、重試與成本。

In [10]:
def interpret_metric(metric_name: str, score: float) -> str:
    descriptions = {
        "faithfulness": (
            "回答中的敘述大多可由檢索內容支持。",
            "回答可能有部分敘述缺少檢索內容支持。",
            "回答有較高的無依據生成風險。",
        ),
        "context_precision": (
            "檢索結果大多與問題相關，相關內容的排序也較理想。",
            "檢索結果混有部分不相關內容，或排序仍可改善。",
            "檢索結果的相關性偏低，應檢查查詢、embedding 或 top_k。",
        ),
        "context_recall": (
            "檢索內容涵蓋了參考答案所需的大部分資訊。",
            "檢索內容只涵蓋部分必要資訊。",
            "檢索內容遺漏較多回答問題所需的資訊。",
        ),
        "factual_correctness": (
            "回答與參考答案在事實上大致一致。",
            "回答與參考答案部分一致，但仍有遺漏或不符。",
            "回答與參考答案的事實一致性偏低。",
        ),
    }

    if pd.isna(score):
        return "無法計算分數，請檢查輸入資料與 evaluator 的輸出。"

    high, medium, low = descriptions[metric_name]
    if score >= 0.80:
        return high
    if score >= 0.50:
        return medium
    return low


async def evaluate_row(row: dict) -> dict:
    faithfulness = await faithfulness_metric.ascore(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    context_precision = await context_precision_metric.ascore(
        user_input=row["user_input"],
        reference=row["reference"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    context_recall = await context_recall_metric.ascore(
        user_input=row["user_input"],
        reference=row["reference"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    factual_correctness = await factual_correctness_metric.ascore(
        response=row["response"],
        reference=row["reference"],
    )

    return {
        **row,
        "faithfulness": float(faithfulness.value),
        "faithfulness_interpretation": interpret_metric(
            "faithfulness", float(faithfulness.value)
        ),
        "context_precision": float(context_precision.value),
        "context_precision_interpretation": interpret_metric(
            "context_precision", float(context_precision.value)
        ),
        "context_recall": float(context_recall.value),
        "context_recall_interpretation": interpret_metric(
            "context_recall", float(context_recall.value)
        ),
        "factual_correctness": float(factual_correctness.value),
        "factual_correctness_interpretation": interpret_metric(
            "factual_correctness", float(factual_correctness.value)
        ),
    }

In [11]:
scored_rows = []

for row in evaluation_rows:
    scored_rows.append(await evaluate_row(row))

score_df = pd.DataFrame(scored_rows)
metric_columns = [
    "faithfulness",
    "context_precision",
    "context_recall",
    "factual_correctness",
]

display_columns = [
    "user_input",
    "faithfulness",
    "faithfulness_interpretation",
    "context_precision",
    "context_precision_interpretation",
    "context_recall",
    "context_recall_interpretation",
    "factual_correctness",
    "factual_correctness_interpretation",
]

pd.set_option("display.max_colwidth", None)
display(score_df[display_columns])

,user_input,faithfulness,faithfulness_interpretation,context_precision,context_precision_interpretation,context_recall,context_recall_interpretation,factual_correctness,factual_correctness_interpretation
0,MemorySaver 的資料會永久保存嗎？,1.00,回答中的敘述大多可由檢索內容支持。,1.0,檢索結果大多與問題相關，相關內容的排序也較理想。,1.0,檢索內容涵蓋了參考答案所需的大部分資訊。,0.86,回答與參考答案在事實上大致一致。
1,如何讓 LangGraph 在發布前等待人工同意？,1.00,回答中的敘述大多可由檢索內容支持。,0.5,檢索結果的相關性偏低，應檢查查詢、embedding 或 top_k。,1.0,檢索內容涵蓋了參考答案所需的大部分資訊。,0.80,回答與參考答案在事實上大致一致。
2,RAGAS 和 MLflow 在評估流程中分別做什麼？,0.75,回答可能有部分敘述缺少檢索內容支持。,1.0,檢索結果大多與問題相關，相關內容的排序也較理想。,1.0,檢索內容涵蓋了參考答案所需的大部分資訊。,0.67,回答與參考答案部分一致，但仍有遺漏或不符。


## 10. 查看平均分數與低分案例

平均分數適合比較不同實驗，但不能取代逐題分析。相同平均值可能來自完全不同的失敗分布。

In [12]:
average_metrics = {
    metric: float(score_df[metric].mean())
    for metric in metric_columns
}

print("Average metrics:")
for metric, value in average_metrics.items():
    print(f"- {metric}: {value:.3f}")

Average metrics:
- faithfulness: 0.917
- context_precision: 0.833
- context_recall: 1.000
- factual_correctness: 0.777


In [13]:
LOW_SCORE_THRESHOLD = 0.80

low_score_mask = (score_df[metric_columns] < LOW_SCORE_THRESHOLD).any(axis=1)
low_score_cases = score_df.loc[
    low_score_mask,
    ["user_input", "response", *metric_columns],
]

display(low_score_cases)

,user_input,response,faithfulness,context_precision,context_recall,factual_correctness
1,如何讓 LangGraph 在發布前等待人工同意？,要讓 LangGraph 在發布前等待人工同意，可以在敏感節點使用 `interrupt_before`。這樣會在執行前暫停流程，等待人工同意。人工同意後，使用相同的 `thread_id` 和 `input=None` 來恢復流程。,1.00,0.5,1.0,0.80
2,RAGAS 和 MLflow 在評估流程中分別做什麼？,RAGAS 主要負責評估品質，而 MLflow 則用來保存實驗參數、分數、artifacts 與 traces。MLflow Tracing 可以記錄 Agent 請求中的 LLM、Tool 與 Node spans。這兩者在評估流程中各自扮演不同的角色。,0.75,1.0,1.0,0.67


### 如何根據低分找問題

- Context Precision 低：可能 `k` 太大、chunk 太雜，或 embedding／query 不適合。
- Context Recall 低：可能 `k` 太小、切塊不完整，或知識庫根本缺資料。
- Faithfulness 低：模型加入了 context 沒有的內容，應加強 grounding prompt 或換模型。
- Factual Correctness 低但 Faithfulness 高：模型可能忠實引用了錯誤／不完整的 context，問題偏向資料或檢索。

## 11. 練習：故意加入 Hallucination

把 context 沒有提到的說法加入回答，再比較 Faithfulness。這能幫助理解指標，而不只是把 RAGAS 當成黑盒。

In [14]:
hallucinated_response = (
    sample["response"]
    + " MemorySaver 也會自動將所有資料永久備份到雲端。"
)

hallucinated_result = await faithfulness_metric.ascore(
    user_input=sample["user_input"],
    response=hallucinated_response,
    retrieved_contexts=sample["retrieved_contexts"],
)

print("Original faithfulness:    ", faithfulness_result.value)
print("Hallucinated faithfulness:", hallucinated_result.value)
print("分數解讀：", interpret_metric("faithfulness", hallucinated_result.value))

Original faithfulness:     1.0
Hallucinated faithfulness: 0.8
分數解讀： 回答中的敘述大多可由檢索內容支持。


## 12. 品質門檻（Quality Gate）

部署前可以設定最低要求。門檻應根據產品風險與實際基準決定，不應直接照抄範例數字。

In [15]:
quality_thresholds = {
    "faithfulness": 0.80,
    "context_precision": 0.75,
    "context_recall": 0.80,
    "factual_correctness": 0.75,
}

failed_metrics = {
    metric: {"score": average_metrics[metric], "threshold": threshold}
    for metric, threshold in quality_thresholds.items()
    if average_metrics[metric] < threshold
}

if failed_metrics:
    print("Quality gate: FAILED")
    for metric, detail in failed_metrics.items():
        print(
            f"- {metric}: {detail['score']:.3f} "
            f"< {detail['threshold']:.3f}"
        )
else:
    print("Quality gate: PASSED")

Quality gate: PASSED


# 接到 MLflow 後要做什麼？

RAGAS 與 MLflow 解決不同問題：

| 工具 | 回答的問題 |
|---|---|
| RAGAS | 這次 RAG 的 retrieval 與 generation 品質好不好？ |
| MLflow Runs | 哪一組模型、`top_k`、prompt 與 embedding 產生這些分數？ |
| MLflow Artifacts | 哪些逐題結果與分數解讀需要保存？ |
| MLflow Tracing | 某個低分案例實際經過哪些 LLM、Retriever 或 Tool 步驟？ |

典型流程是：

1. 修改一項 RAG 設定，例如 `TOP_K=1` 或 `TOP_K=3`。
2. 對固定 test set 重新產生回答。
3. 使用 RAGAS 計算相同 metrics。
4. 用 MLflow Run 記錄參數與平均分數。
5. 將逐題分數、回答與診斷資訊存成 artifact。
6. 在 MLflow UI 比較 Runs，找出改善或退步。
7. 對低分案例查看 Trace，定位是 retrieval 還是 generation 問題。

## 13. 可選：將本次 RAGAS 結果記錄到 MLflow

請先在另一個 terminal 啟動 MLflow：

```powershell
mlflow server --host 127.0.0.1 --port 5000
```

下面預設 `ENABLE_MLFLOW=False`，不會主動連線。確認 server 已啟動後再改為 `True`。

In [17]:
ENABLE_MLFLOW = True

if ENABLE_MLFLOW:
    import mlflow

    mlflow.set_tracking_uri(
        os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
    )
    mlflow.set_experiment("ragas-rag-course")

    with mlflow.start_run(run_name=f"ragas-top-k-{TOP_K}"):
        mlflow.log_params(
            {
                "rag_model": rag_model_name,
                "embedding_model": embedding_model_name,
                "evaluator_model": evaluator_model_name,
                "top_k": TOP_K,
                "test_case_count": len(test_cases),
                "ragas_version": ragas.__version__,
            }
        )
        mlflow.log_metrics(average_metrics)
        mlflow.log_table(
            data=score_df,
            artifact_file="evaluation/ragas_results.json",
        )
        mlflow.log_dict(
            quality_thresholds,
            "evaluation/quality_thresholds.json",
        )

    print("Results logged to:", mlflow.get_tracking_uri())
else:
    print("MLflow logging skipped. Set ENABLE_MLFLOW=True when the server is ready.")

2026/09/25 16:11:54 INFO mlflow.tracking.fluent: Experiment with name 'ragas-rag-course' does not exist. Creating a new experiment.


🏃 View run ragas-top-k-2 at: http://127.0.0.1:5000/#/experiments/3/runs/1cfeea59d5e54169b066a951414938e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Results logged to: http://127.0.0.1:5000


## 14. 如果還要記錄 Traces

上一格記錄的是實驗層級資料：params、平均 metrics 與逐題 artifact。若還想查看每一題的 Retriever／LLM 執行細節，可以在產生 evaluation rows **之前** 開啟：

```python
mlflow.langchain.autolog(log_traces=True)
```

建議區分兩種資料：

- RAG application trace：用來找出 retrieval、prompt 或 generation 的失敗位置。
- RAGAS evaluator calls：評分模型自己的呼叫，可能很多且成本較高。

如果所有 judge calls 都被 autolog，UI 可能變得很雜。實務上可只追蹤 `run_rag()`，或將應用 Trace 與 Evaluation Run 放到不同 experiments／tags。

## 練習題

1. 將 `TOP_K` 改為 1、2、3，哪個設定的 Context Precision 與 Recall 最好？
2. 在知識庫加入一段不相關內容，觀察 Context Precision。
3. 移除 Human-in-the-Loop 文件，觀察該題 Context Recall。
4. 修改 prompt，允許模型自由補充常識，觀察 Faithfulness。
5. 改用不同 RAG model 或 evaluator model，比較分數和成本。
6. 把每次設定記錄成 MLflow Run，確認改善不是只發生在單一題目。

## 重要限制

- RAGAS 分數依賴 evaluator model，不同 judge 可能給不同結果。
- 小型測試集的平均分數不代表正式流量品質。
- Reference 品質錯誤會直接污染評估。
- LLM-based metrics 有 token、延遲與 rate-limit 成本。
- 自動指標應搭配人工抽查、領域測試與安全測試。

延伸閱讀：

- [RAGAS LangChain integration](https://docs.ragas.io/en/latest/howtos/integrations/langchain/)
- [RAGAS 0.3 → 0.4 migration](https://docs.ragas.io/en/stable/howtos/migrations/migrate_from_v03_to_v04/)
- [MLflow RAGAS integration](https://www.mlflow.org/docs/latest/genai/eval-monitor/scorers/third-party/ragas/)